# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [2]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [3]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer
License(s): ODbL-1.0


100%|█████████████████████████████████████| 32.7M/32.7M [00:02<00:00, 13.8MB/s]



Connected to database at: /var/folders/jl/9dx0t8f14zxgdhb7jmsg80wh0000gn/T/database.sqlite


---

## SQL Exploration

In [4]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [5]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Match LIMIT 5", connection)
display(df.head())

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67


In [6]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Team", connection)
display(df['team_long_name'].unique())

array(['KRC Genk', 'Beerschot AC', 'SV Zulte-Waregem', 'Sporting Lokeren',
       'KSV Cercle Brugge', 'RSC Anderlecht', 'KAA Gent', 'RAEC Mons',
       'FCV Dender EH', 'Standard de Liège', 'KV Mechelen',
       'Club Brugge KV', 'KSV Roeselare', 'KV Kortrijk', 'Tubize',
       'Royal Excel Mouscron', 'KVC Westerlo', 'Sporting Charleroi',
       'Sint-Truidense VV', 'Lierse SK', 'KAS Eupen',
       'Oud-Heverlee Leuven', 'Waasland-Beveren', 'KV Oostende',
       'Manchester United', 'Newcastle United', 'Arsenal',
       'West Bromwich Albion', 'Sunderland', 'Liverpool',
       'West Ham United', 'Wigan Athletic', 'Aston Villa',
       'Manchester City', 'Everton', 'Blackburn Rovers', 'Middlesbrough',
       'Tottenham Hotspur', 'Bolton Wanderers', 'Stoke City', 'Hull City',
       'Fulham', 'Chelsea', 'Portsmouth', 'Birmingham City',
       'Wolverhampton Wanderers', 'Burnley', 'Blackpool', 'Swansea City',
       'Queens Park Rangers', 'Norwich City', 'Southampton', 'Reading',
       

In [30]:
df = pd.read_sql('SELECT * FROM Player_Attributes', connection)
display(df.info())
df.columns
display(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183978 entries, 0 to 183977
Data columns (total 42 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   183978 non-null  int64  
 1   player_fifa_api_id   183978 non-null  int64  
 2   player_api_id        183978 non-null  int64  
 3   date                 183978 non-null  object 
 4   overall_rating       183142 non-null  float64
 5   potential            183142 non-null  float64
 6   preferred_foot       183142 non-null  object 
 7   attacking_work_rate  180748 non-null  object 
 8   defensive_work_rate  183142 non-null  object 
 9   crossing             183142 non-null  float64
 10  finishing            183142 non-null  float64
 11  heading_accuracy     183142 non-null  float64
 12  short_passing        183142 non-null  float64
 13  volleys              181265 non-null  float64
 14  dribbling            183142 non-null  float64
 15  curve            

None

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183973,183974,102359,39902,2009-08-30 00:00:00,83.0,85.0,right,medium,low,84.0,...,88.0,83.0,22.0,31.0,30.0,9.0,20.0,84.0,20.0,20.0
183974,183975,102359,39902,2009-02-22 00:00:00,78.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183975,183976,102359,39902,2008-08-30 00:00:00,77.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183976,183977,102359,39902,2007-08-30 00:00:00,78.0,81.0,right,medium,low,74.0,...,88.0,53.0,28.0,32.0,30.0,9.0,20.0,73.0,20.0,20.0


Is de tabel relevant?
ja, het is relevant 
Wat zijn de kolommen, welke zijn interessant?

Wat betekenen de classifiers in de kolommen?

Wat zijn de overeenkomende kolommen?

# Beschrijving van de Player_Attributes table:

| Column              	| Beschrijving                                                                                                                                                                                           	| Relevantie 	| sort data 	|
|---------------------	|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------	|------------	|-----------	|
| id                  	| dit is de id van de players.                                                                                                                                                                           	| 1          	| integer   	|
| player_fifa_api_id  	| het unieke nummer dat een speler identificeert in de FIFA-spelgegevens.                                                                                                                                	| 0          	| integer   	|
| player_api_id       	| Een uniek nummer dat een speler identificeert wanneer gegevens via een API of database worden gebruikt. API : Application Programming Interface (een systeem waarmee software met elkaar communiceert) 	| 0          	| integer   	|
| date                	| de date wanneer de data is verzameld.                                                                                                                                                                  	| 0          	| object    	|
| overall_rating      	| rating van de players.                                                                                                                                                                                 	| 1          	| float     	|
| potential           	| de potentieel die de player heeft.                                                                                                                                                                     	| 1          	| float     	|
| preferred_foot      	| geven of de speler rechter been is of linker. ( recht of links ).                                                                                                                                      	| 1          	| object    	|
| attacking_work_rate 	| hoe vaak en hoe intens een speler naar voren beweegt om aan te vallen.                                                                                                                                 	| 1          	| object    	|
| defensive_work_rate 	| hoe vaak en hoe intens een speler terugloopt om te verdedigen en het team te helpen zonder bal.                                                                                                        	| 1          	| object    	|
| crossing            	| de vaardigheid van een speler om een nauwkeurige voorzet naar een teamgenoot te geven.                                                                                                                 	| 1          	| float     	|
| finishing           	| de vaardigheid van een speler om een schot om te zetten in een doelpunt.                                                                                                                               	| 1          	| float     	|
| heading_accuracy    	| hoe nauwkeurig een speler de bal met het hoofd kan raken en richting doel of een teamgenoot kan sturen.                                                                                                	| 1          	| float     	|
| short_passing       	| hoe goed en nauwkeurig een speler korte passes naar teamgenoten kan geven.                                                                                                                             	| 1          	| float     	|
| volleys             	| hoe goed een speler de bal kan raken en afmaken terwijl deze in de lucht is, zonder dat de bal eerst de grond raakt.                                                                                   	| 1          	| float     	|
| dribbling           	| hoe goed een speler de bal kan controleren en met snelheid en behendigheid langs tegenstanders kan bewegen.                                                                                            	| 1          	| float     	|
| curve               	| hoe goed een speler de bal kan laten draaien of buigen tijdens een schot of pass.                                                                                                                      	| 1          	| float     	|
| free_kick_accuracy  	| hoe nauwkeurig een speler vrije trappen kan nemen richting doel of teamgenoten.                                                                                                                        	| 1          	| float     	|
| long_passing        	| hoe goed een speler lange passes over grote afstand nauwkeurig naar teamgenoten kan spelen.                                                                                                            	| 1          	| float     	|
| ball_control        	| hoe goed een speler de bal kan beheersen bij ontvangst, zodat hij snel kan passen, dribbelen of schieten.                                                                                              	| 1          	| float     	|
| acceleration        	| hoe snel een speler van stilstand naar maximale snelheid kan versnellen.                                                                                                                               	| 1          	| float     	|
| sprint_speed        	| de maximale snelheid die een speler kan halen tijdens een sprint.                                                                                                                                      	| 1          	| float     	|
| agility             	| hoe wendbaar een speler is, oftewel hoe snel hij van richting kan veranderen terwijl hij de bal beheerst.                                                                                              	| 1          	| float     	|
| reactions           	| hoe snel een speler kan reageren op situaties tijdens het spel, zoals kansen, passes of tackles.                                                                                                       	| 1          	| float     	|
| balance             	| hoe goed een speler zijn lichaam stabiel kan houden tijdens beweging, tackles of dribbels.                                                                                                             	| 1          	| float     	|
| shot_power          	| hoe hard een speler de bal kan schieten bij een schot op doel.                                                                                                                                         	| 1          	| float     	|
| jumping             	| hoe hoog een speler kan springen, bijvoorbeeld bij kopballen of duels in de lucht.                                                                                                                     	| 1          	| float     	|
| stamina             	| hoe goed een speler zijn energie behoudt gedurende de hele wedstrijd en hoe lang hij intensief kan blijven spelen.                                                                                     	| 1          	| float     	|
| strength            	| hoe fysiek krachtig een speler is, bijvoorbeeld bij duels, het vasthouden van de bal of het afweren van tegenstanders.                                                                                 	| 1          	| float     	|
| long_shots          	| hoe goed een speler van afstand kan schieten en doelpunten kan maken van buiten het strafschopgebied.                                                                                                  	| 1          	| float     	|
| aggression          	| hoe intens een speler optreedt in duels, tackels en het opdrukken van tegenstanders tijdens het spel.                                                                                                  	| 1          	| float     	|
| interceptions       	| hoe goed een speler passes van tegenstanders kan onderscheppen en de bal kan veroveren.                                                                                                                	| 1          	| float     	|
| positioning         	| hoe goed een speler zich op het juiste moment op de juiste plek op het veld positioneert, vooral bij aanvallen of verdedigende acties.                                                                 	| 1          	| float     	|
| vision              	| hoe goed een speler het spel kan overzien en slimme passes kan maken naar teamgenoten.                                                                                                                 	| 1          	| float     	|
| penalties           	| hoe nauwkeurig en effectief een speler strafschoppen kan nemen en scoren.                                                                                                                              	| 1          	| float     	|
| marking             	| hoe goed een speler een tegenstander kan volgen en de ruimte beperkt om kansen te voorkomen.                                                                                                           	| 1          	| float     	|
| standing_tackle     	| hoe goed een speler een tegenstander kan tackelen terwijl hij zelf rechtop blijft staan.                                                                                                               	| 1          	| float     	|
| sliding_tackle      	| hoe goed een speler een tegenstander kan tackelen door een schuifbeweging over de grond te maken.                                                                                                      	| 1          	| float     	|
| gk_diving           	| hoe goed een doelman (goalkeeper) kan duiken om schoten op doel te stoppen.                                                                                                                            	| 1          	| float     	|
| gk_handling         	| hoe goed een doelman de bal kan vastpakken, controleren en vasthouden tijdens schoten of voorzetten.                                                                                                   	| 1          	| float     	|
| gk_kicking          	| hoe krachtig en nauwkeurig een doelman de bal kan wegtrappen of wegschieten naar teamgenoten.                                                                                                          	| 1          	| float     	|
| gk_positioning      	| hoe goed een doelman zich op de juiste plek kan positioneren om schoten op doel te blokkeren of het doel te beschermen.                                                                                	| 1          	| float     	|
| gk_reflexes         	| hoe snel een doelman kan reageren op onverwachte schoten of acties om een doelpunt te voorkomen.                                                                                                       	| 1          	| float     	|

In [37]:
df1= pd.read_sql('SELECT * FROM Player', connection)
display(df1.info())
display(df1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11060 entries, 0 to 11059
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  11060 non-null  int64  
 1   player_api_id       11060 non-null  int64  
 2   player_name         11060 non-null  object 
 3   player_fifa_api_id  11060 non-null  int64  
 4   birthday            11060 non-null  object 
 5   height              11060 non-null  float64
 6   weight              11060 non-null  int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 605.0+ KB


None

,id,player_api_id,player_name,player_fifa_api_id,birthday,height,weight
0,1,505942,Aaron Appindangoye,218353,1992-02-29 00:00:00,182.88,187
1,2,155782,Aaron Cresswell,189615,1989-12-15 00:00:00,170.18,146
2,3,162549,Aaron Doran,186170,1991-05-13 00:00:00,170.18,163
3,4,30572,Aaron Galindo,140161,1982-05-08 00:00:00,182.88,198
4,5,23780,Aaron Hughes,17725,1979-11-08 00:00:00,182.88,154
...,...,...,...,...,...,...,...
11055,11071,26357,Zoumana Camara,2488,1979-04-03 00:00:00,182.88,168
11056,11072,111182,Zsolt Laczko,164680,1986-12-18 00:00:00,182.88,176
11057,11073,36491,Zsolt Low,111191,1979-04-29 00:00:00,180.34,154
11058,11074,35506,Zurab Khizanishvili,47058,1981-10-06 00:00:00,185.42,172


| Column             	| Beschrijving                                                                                                                                                                                            	| Relevantie 	| sort data 	|
|--------------------	|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------	|------------	|-----------	|
| id                 	| de id van de spelers.                                                                                                                                                                                   	| 1          	| integer   	|
| player_api_id      	| Een uniek nummer dat een speler identificeert wanneer gegevens via een API of database worden gebruikt.  API : Application Programming Interface (een systeem waarmee software met elkaar communiceert) 	| 0          	| integer   	|
| player_name        	| De naam van de speler.                                                                                                                                                                                  	| 0          	| object    	|
| player_fifa_api_id 	| Het unieke nummer dat een speler identificeert in de FIFA-spelgegevens.                                                                                                                                 	| 0          	| integer   	|
| birthday           	| De geboortedatum van de spelers.                                                                                                                                                                        	| 0          	| object    	|
| height             	| De lengte van de spelers.                                                                                                                                                                               	| 1          	| float     	|
| weight             	| De gewicht van de spelers.                                                                                                                                                                              	| 1          	| integer   	|